In [1]:
from vinschgaumap import coord_at_pk

Punti polyline: 1819
Lunghezza totale: 59.832 km

Lunghezze raddoppi nuovi per scenario:
  Scenario 1a: 14.703 km
  Scenario 1b: 13.722 km
  Scenario 1c: 17.070 km
  Scenario 2: 16.321 km
  Scenario 3: 12.647 km
  Gallerie: 2.485 km

Nodi costo identificati: 25
Segmenti costo creati: 24

Costi stimati per scenario su base costo unitario e km raddoppiati:
  Scenario 1a: 14.703 km, costo totale €618.53
  Scenario 1b: 13.722 km, costo totale €844.88
  Scenario 1c: 17.070 km, costo totale €1154.39
    Attenzione: 0.025 km non coperti da segmenti costo noti
  Scenario 2: 16.321 km, costo totale €985.95
    Attenzione: 0.025 km non coperti da segmenti costo noti
  Scenario 3: 12.647 km, costo totale €431.54
  Gallerie: 2.485 km, costo totale €394.90

Mappa salvata — 266 segnali, 3 scenari.
c:\Users\LeoC\VSCodes\optimizationVinschgau\AusbauVinschgau\tools\geographic_map\val_venosta.html


In [2]:
coord_at_pk(50)

(46.629176383774244, 10.633152566205583)

## Calibrate CSV nodes against the geographic route

Run the next cell, inspect the markers on the map, and adjust only one calibration parameter at a time. The default offset reproduces the `pk_rel - 0.025` correction used by the module's named-node lookup.

In [3]:
import csv
import os
import folium
from IPython.display import IFrame, display
from vinschgaumap import coord_at_pk, ordered_coords, total_length_m

# Scale the complete CSV PK span onto the JSON geometry span.
GEOMETRY_START_KM = 0.0
GEOMETRY_END_KM = None  # None means the complete JSON geometry length
REVERSE_PK = False      # set True if CSV order runs opposite to the route
Y_FILTER = 0.0          # import the mainline track (CSV rows with y == this value)

csv_path = os.path.join(os.path.dirname(os.path.abspath("coord_at_pk.ipynb")), "nodes_mainline.csv")
route_length_km = total_length_m / 1000.0
geometry_end_km = route_length_km if GEOMETRY_END_KM is None else GEOMETRY_END_KM
nodes = []
with open(csv_path, newline="", encoding="utf-8") as csv_file:
    reader = csv.DictReader(csv_file, delimiter=";")
    for row in reader:
        try:
            y_value = float((row.get("y") or "").strip().replace(",", "."))
            pk_rel = float((row.get("pk_rel") or "").strip().replace(",", "."))
        except ValueError:
            continue
        if abs(y_value - Y_FILTER) <= 1e-9:
            nodes.append({
                "name": (row.get("name") or row.get("node_id") or "unnamed").strip(),
                "node_id": (row.get("node_id") or "").strip(),
                "pk_rel": pk_rel,
            })

if not nodes:
    raise ValueError(f"No CSV nodes found for Y_FILTER={Y_FILTER}")

csv_start_km = min(node["pk_rel"] for node in nodes)
csv_end_km = max(node["pk_rel"] for node in nodes)
csv_span_km = csv_end_km - csv_start_km
geometry_span_km = geometry_end_km - GEOMETRY_START_KM
if csv_span_km <= 0:
    raise ValueError("CSV PK span must be greater than zero")
if geometry_span_km <= 0:
    raise ValueError("Geometry PK span must be greater than zero")

for node in nodes:
    fraction = (node["pk_rel"] - csv_start_km) / csv_span_km
    scaled_pk = GEOMETRY_START_KM + fraction * geometry_span_km
    node["route_pk"] = geometry_end_km - scaled_pk if REVERSE_PK else scaled_pk
    node["lat"], node["lon"] = coord_at_pk(node["route_pk"])

route_latlon = [(lat, lon) for lon, lat in ordered_coords]
center = route_latlon[len(route_latlon) // 2]
calibration_map = folium.Map(location=center, zoom_start=11, tiles="OpenStreetMap")
route_layer = folium.FeatureGroup(name="JSON route", show=True).add_to(calibration_map)
folium.PolyLine(route_latlon, color="#d1495b", weight=5, opacity=0.85).add_to(route_layer)
node_layer = folium.FeatureGroup(name="CSV nodes", show=True).add_to(calibration_map)
for node in nodes:
    folium.CircleMarker(
        location=(node["lat"], node["lon"]), radius=4, color="#00798c",
        fill=True, fill_opacity=0.9,
        tooltip=f'{node["name"]} | CSV PK {node["pk_rel"]:.3f} | scaled PK {node["route_pk"]:.3f}',
    ).add_to(node_layer)
folium.LayerControl().add_to(calibration_map)
calibration_map.fit_bounds([route_latlon[0], route_latlon[-1]])
output_path = os.path.join(os.path.dirname(os.path.abspath("coord_at_pk.ipynb")), "coord_at_pk_calibration.html")
calibration_map.save(output_path)

direction = "reversed" if REVERSE_PK else "forward"
scale_factor = geometry_span_km / csv_span_km
print(f"Loaded {len(nodes)} nodes from {csv_path}")
print(f"CSV PK span: {csv_start_km:.3f} to {csv_end_km:.3f} km ({csv_span_km:.3f} km)")
print(f"JSON geometry span: {GEOMETRY_START_KM:.3f} to {geometry_end_km:.3f} km ({geometry_span_km:.3f} km)")
print(f"Applied PK scale factor: {scale_factor:.9f}, direction {direction}")
print("Basemap: OpenStreetMap")
print(f"Map saved to {output_path}")
display(IFrame(src="coord_at_pk_calibration.html", width="100%", height=700))

print("Tune these parameters:")
print("  GEOMETRY_START_KM / GEOMETRY_END_KM: choose the JSON route interval receiving the full CSV distribution.")
print("  REVERSE_PK: direction; use True when the first CSV node belongs at the route's Malles end.")
print("  Y_FILTER: track selection; change only if a different CSV y track is intended.")

Loaded 266 nodes from c:\Users\LeoC\VSCodes\optimizationVinschgau\AusbauVinschgau\tools\geographic_map\nodes_mainline.csv
CSV PK span: 0.000 to 59.875 km (59.875 km)
JSON geometry span: 0.000 to 59.832 km (59.832 km)
Applied PK scale factor: 0.999285218, direction forward
Basemap: OpenStreetMap
Map saved to c:\Users\LeoC\VSCodes\optimizationVinschgau\AusbauVinschgau\tools\geographic_map\coord_at_pk_calibration.html


Tune these parameters:
  GEOMETRY_START_KM / GEOMETRY_END_KM: choose the JSON route interval receiving the full CSV distribution.
  REVERSE_PK: direction; use True when the first CSV node belongs at the route's Malles end.
  Y_FILTER: track selection; change only if a different CSV y track is intended.
